## ПРАКТИЧЕСКАЯ РАБОТА №2.
«ИССЛЕДОВАНИЕ ДАННЫХ НА PYTHON. ОБРАБОТКА ВЫБРОСОВ.
ВИЗУАЛЬНЫЙ EDA»

In [1172]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings


In [1173]:
def get_real_value(nom, old, new_cpi):
    return (nom * new_cpi) / old

## Загрузка данных


In [1174]:
data = pd.read_csv("Dataset1.csv")
cpi_table = pd.read_csv("Practice2_Harlov_CPI.csv")
cpi_2016 = float(cpi_table[cpi_table["year"]==2016]["avg_cpi"].values[0])

In [1175]:
cpi_table = cpi_table.drop(columns="Unnamed: 0")

cpi_table

,year,avg_cpi
0,1913,9.900
1,1914,10.000
2,1915,10.100
3,1916,10.900
4,1917,12.800
...,...,...
99,2012,229.594
100,2013,232.957
101,2014,236.736
102,2015,237.017


## Предобработка данных

In [1176]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      5025 non-null   str    
 1   Director_Name              4872 non-null   str    
 2   num_Critic_for_reviews     4927 non-null   float64
 3   duration                   4959 non-null   float64
 4   director_Facebook_likes    4872 non-null   float64
 5   actor_3_Facebook_likes     4953 non-null   float64
 6   actor_2_name               4963 non-null   str    
 7   Actor_1_Facebook_likes     4968 non-null   float64
 8   gross                      4104 non-null   float64
 9   genres                     4974 non-null   str    
 10  actor_1_name               4968 non-null   str    
 11  movie_Title                4974 non-null   str    
 12  num_voted_users            4974 non-null   float64
 13  cast_total_facebook_likes  4974 non-null   float64
 14  act

In [1177]:
data.median(numeric_only=True)

num_Critic_for_reviews            110.00
duration                          103.00
director_Facebook_likes            49.00
actor_3_Facebook_likes            372.00
Actor_1_Facebook_likes            989.00
gross                        25591375.50
num_voted_users                 34504.00
cast_total_facebook_likes        3097.50
facenumber_in_poster                1.00
num_user_for_reviews              157.00
budget                       20000000.00
title_year                       2005.00
actor_2_facebook_likes            595.00
imdb_score                          6.60
aspect_ratio                        2.35
dtype: float64

### Мы видим что датасет:
- состоит из 28 столбцов
- в нем 5043 строки
- данные представленны string и float

In [1178]:
data.head(5)

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000;
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0;
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000;
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000;
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0;


## План на очистку данных 
- убрать ошибку представения датасета, когда все данные были записанны в одну ячейку
- найти и удалить дубликаты
- устранить пропуски

#### Удаление полностью пустых строк

In [1179]:
old_data_size = data.shape[0]
data = data.drop(data[data.isna().sum(axis=1) == 27].index).reset_index(drop=True)
deleted = old_data_size - data.shape[0]
deleted

69

Мы избавились от 69 полность пустых строк 

#### Теперь переходим к удалению дубликатов

In [1180]:
data.duplicated().sum()

np.int64(44)

In [1181]:
data = data.drop_duplicates(ignore_index=True)

Мы удалили полные дубликаты данных

In [1182]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4930 entries, 0 to 4929
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      4912 non-null   str    
 1   Director_Name              4829 non-null   str    
 2   num_Critic_for_reviews     4884 non-null   float64
 3   duration                   4915 non-null   float64
 4   director_Facebook_likes    4829 non-null   float64
 5   actor_3_Facebook_likes     4909 non-null   float64
 6   actor_2_name               4919 non-null   str    
 7   Actor_1_Facebook_likes     4924 non-null   float64
 8   gross                      4070 non-null   float64
 9   genres                     4930 non-null   str    
 10  actor_1_name               4924 non-null   str    
 11  movie_Title                4930 non-null   str    
 12  num_voted_users            4930 non-null   float64
 13  cast_total_facebook_likes  4930 non-null   float64
 14  act

In [1183]:
data.duplicated(subset="movie_Title").sum()

np.int64(80)

#### Здесь мы видим: 
- у нас есть строки которые описывают фильмы, которые уже есть в датасете
- в датасете отображены данные из одного источника и они не должны повторяться тк каждому фильму может быть выставленна только одна оценка 

Я решил что такие дубликаты скорее всего отлицаются в какой-то одной переменной на величину погрешности 

In [1184]:
data = data.drop_duplicates(subset="movie_Title",ignore_index=True)

In [1185]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4850 entries, 0 to 4849
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      4832 non-null   str    
 1   Director_Name              4750 non-null   str    
 2   num_Critic_for_reviews     4804 non-null   float64
 3   duration                   4835 non-null   float64
 4   director_Facebook_likes    4750 non-null   float64
 5   actor_3_Facebook_likes     4829 non-null   float64
 6   actor_2_name               4839 non-null   str    
 7   Actor_1_Facebook_likes     4844 non-null   float64
 8   gross                      4000 non-null   float64
 9   genres                     4850 non-null   str    
 10  actor_1_name               4844 non-null   str    
 11  movie_Title                4850 non-null   str    
 12  num_voted_users            4850 non-null   float64
 13  cast_total_facebook_likes  4850 non-null   float64
 14  act

Мы разобрались со всеми возможными дубликатами

#### Приступаем к обработке пропусков

In [1186]:
(data.isna().sum()/data.shape[0]*100).sort_values(ascending=False)

gross                        17.525773
budget                        9.752577
aspect_ratio                  6.536082
content_rating                6.061856
plot_keywords                 3.010309
title_year                    2.123711
Director_Name                 2.061856
director_Facebook_likes       2.061856
num_Critic_for_reviews        0.948454
actor_3_name                  0.432990
actor_3_Facebook_likes        0.432990
num_user_for_reviews          0.371134
color                         0.371134
duration                      0.309278
facenumber_in_poster          0.268041
language                      0.268041
actor_2_name                  0.226804
actor_2_facebook_likes        0.226804
actor_1_name                  0.123711
Actor_1_Facebook_likes        0.123711
country                       0.061856
cast_total_facebook_likes     0.000000
num_voted_users               0.000000
movie_Title                   0.000000
movie_imdb_link               0.000000
genres                   

## title_year

##### Сначала разберемся с годом выапуска фильма, поскольку это признак нам поможет при заполнении других данных боллее точно на основании года их производства

In [1187]:
data["title_year"].isna().sum()

np.int64(103)

У нас есть целых 103 фильма без года его производства, и в их названии нет указания года которое могло бы нам помочь.
- мы можем попробовать заполнить год выпуска, по медиане для конкретного режиссера

In [1188]:
data[data["title_year"].isna() & data["Director_Name"].isna()].shape[0]


100

##### Мы видим что почти во всех строках где пропущен режисер, пропущен и год, это значит, что мы можем заполнить только медианой

In [1189]:
data.fillna({"title_year": data['title_year'].median()}, inplace=True)

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000;
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0;
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000;
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000;
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,2005.0,12.0,7.1,NaN,0;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4845,Color,Scott Smith,1.0,87.0,2.0,318.0,Daphne Zuniga,637.0,NaN,Comedy|Drama,...,6.0,English,Canada,NaN,NaN,2013.0,470.0,7.7,NaN,84;
4846,Color,NaN,43.0,43.0,NaN,319.0,Valorie Curry,841.0,NaN,Crime|Drama|Mystery|Thriller,...,359.0,English,USA,TV-14,NaN,2005.0,593.0,7.5,16.00,32000;
4847,Color,Benjamin Roberds,13.0,76.0,0.0,0.0,Maxwell Moody,0.0,NaN,Drama|Horror|Thriller,...,3.0,English,USA,NaN,1400.0,2013.0,0.0,6.3,NaN,16;
4848,Color,Daniel Hsia,14.0,100.0,0.0,489.0,Daniel Henney,946.0,10443.0,Comedy|Drama|Romance,...,9.0,English,USA,PG-13,NaN,2012.0,719.0,6.3,2.35,660;


In [1190]:
print(data['title_year'].isna().sum())

0


### Теперь признаки у кторорых <2%

### actor_name and actor likes

In [1191]:
actor_name_cols = ['actor_3_name', 'actor_2_name', 'actor_1_name']

for col in actor_name_cols:
    print(data[col].isna().sum())

21
11
6


In [1192]:
actor_likes_cols = ['actor_3_Facebook_likes', 'actor_2_facebook_likes', 'Actor_1_Facebook_likes']

for col in actor_likes_cols:
    print(data[col].isna().sum())

21
11
6


У нас одинаковое количество пропусков, возможно где пропущенно имя там пропущенны и лайки

In [1193]:
for i in range(3):
    print((data[actor_name_cols[i]].isna() & data[actor_likes_cols[i]].isna()).sum())

21
11
6


Мы видим, что пропуски именно в одних и тех же строках, значит где пропущенно имя мы меняем на unknown а количество лайков на 0

In [1194]:
for col in actor_likes_cols:
    data.fillna({col :0}, inplace=True)
for col in actor_name_cols:
    data.fillna({col :'unknown'}, inplace=True)

print(data[actor_name_cols + actor_likes_cols].isna().sum())

actor_3_name              0
actor_2_name              0
actor_1_name              0
actor_3_Facebook_likes    0
actor_2_facebook_likes    0
Actor_1_Facebook_likes    0
dtype: int64


Убрали

In [1195]:
(data.isna().sum()/data.shape[0]*100).sort_values(ascending=False)

gross                        17.525773
budget                        9.752577
aspect_ratio                  6.536082
content_rating                6.061856
plot_keywords                 3.010309
director_Facebook_likes       2.061856
Director_Name                 2.061856
num_Critic_for_reviews        0.948454
num_user_for_reviews          0.371134
color                         0.371134
duration                      0.309278
language                      0.268041
facenumber_in_poster          0.268041
country                       0.061856
imdb_score                    0.000000
actor_2_facebook_likes        0.000000
title_year                    0.000000
actor_3_name                  0.000000
movie_imdb_link               0.000000
cast_total_facebook_likes     0.000000
num_voted_users               0.000000
movie_Title                   0.000000
actor_1_name                  0.000000
genres                        0.000000
Actor_1_Facebook_likes        0.000000
actor_2_name             

Медианой заменим пропуски:
- facenumber_in_poster
- duration
- num_Critic_for_reviews
- num_user_for_reviewsч

In [1196]:
num_cols = ['num_Critic_for_reviews', 'num_user_for_reviews', 'duration', 'facenumber_in_poster']
for col in num_cols:
    data = data.fillna({col: data[col].median()}, inplace=True)

In [1197]:
print(data[num_cols].isna().sum())

num_Critic_for_reviews    0
num_user_for_reviews      0
duration                  0
facenumber_in_poster      0
dtype: int64


Модой заполним:
- color
- language
- country

In [1198]:
mode_columns = ["color", "language", "country"]

for col in mode_columns:
    data.fillna({col: data[col].mode()[0]}, inplace=True)



In [1199]:
print(data[mode_columns].isna().sum())

color       0
language    0
country     0
dtype: int64


In [1200]:
(data.isna().sum()/data.shape[0]*100).sort_values(ascending=False)

gross                        17.525773
budget                        9.752577
aspect_ratio                  6.536082
content_rating                6.061856
plot_keywords                 3.010309
director_Facebook_likes       2.061856
Director_Name                 2.061856
color                         0.000000
imdb_score                    0.000000
actor_2_facebook_likes        0.000000
title_year                    0.000000
country                       0.000000
language                      0.000000
num_user_for_reviews          0.000000
movie_imdb_link               0.000000
actor_3_name                  0.000000
facenumber_in_poster          0.000000
cast_total_facebook_likes     0.000000
num_voted_users               0.000000
movie_Title                   0.000000
actor_1_name                  0.000000
genres                        0.000000
Actor_1_Facebook_likes        0.000000
actor_2_name                  0.000000
actor_3_Facebook_likes        0.000000
duration                 

### Остались признаки где больше 2% пропусков

director_Facebook_likes
- Сначала заполняем лайки внутри групп по режиссёру (где имя есть)
- Оставшиеся (где и имени нет) — общей медианой или 0

In [1201]:

data['director_Facebook_likes'] = data.groupby('Director_Name')['director_Facebook_likes'].transform(
    lambda x: x.fillna(x.median())
)

data.fillna({'director_Facebook_likes': data['director_Facebook_likes'].median()}, inplace=True)

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000;
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0;
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000;
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000;
4,Color,Doug Walker,108.0,103.0,131.0,0.0,Rob Walker,131.0,NaN,Documentary,...,154.0,English,USA,NaN,NaN,2005.0,12.0,7.1,NaN,0;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4845,Color,Scott Smith,1.0,87.0,2.0,318.0,Daphne Zuniga,637.0,NaN,Comedy|Drama,...,6.0,English,Canada,NaN,NaN,2013.0,470.0,7.7,NaN,84;
4846,Color,NaN,43.0,43.0,49.0,319.0,Valorie Curry,841.0,NaN,Crime|Drama|Mystery|Thriller,...,359.0,English,USA,TV-14,NaN,2005.0,593.0,7.5,16.00,32000;
4847,Color,Benjamin Roberds,13.0,76.0,0.0,0.0,Maxwell Moody,0.0,NaN,Drama|Horror|Thriller,...,3.0,English,USA,NaN,1400.0,2013.0,0.0,6.3,NaN,16;
4848,Color,Daniel Hsia,14.0,100.0,0.0,489.0,Daniel Henney,946.0,10443.0,Comedy|Drama|Romance,...,9.0,English,USA,PG-13,NaN,2012.0,719.0,6.3,2.35,660;


С этими пунктами как с актерами лучше заполнить unknown:
- Director_Name
- plot_keywords

In [1202]:
data.fillna({'Director_Name': 'Unknown'}, inplace=True)
data.fillna({'plot_keywords': 'unknown'}, inplace=True)  

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000;
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0;
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000;
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000;
4,Color,Doug Walker,108.0,103.0,131.0,0.0,Rob Walker,131.0,NaN,Documentary,...,154.0,English,USA,NaN,NaN,2005.0,12.0,7.1,NaN,0;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4845,Color,Scott Smith,1.0,87.0,2.0,318.0,Daphne Zuniga,637.0,NaN,Comedy|Drama,...,6.0,English,Canada,NaN,NaN,2013.0,470.0,7.7,NaN,84;
4846,Color,Unknown,43.0,43.0,49.0,319.0,Valorie Curry,841.0,NaN,Crime|Drama|Mystery|Thriller,...,359.0,English,USA,TV-14,NaN,2005.0,593.0,7.5,16.00,32000;
4847,Color,Benjamin Roberds,13.0,76.0,0.0,0.0,Maxwell Moody,0.0,NaN,Drama|Horror|Thriller,...,3.0,English,USA,NaN,1400.0,2013.0,0.0,6.3,NaN,16;
4848,Color,Daniel Hsia,14.0,100.0,0.0,489.0,Daniel Henney,946.0,10443.0,Comedy|Drama|Romance,...,9.0,English,USA,PG-13,NaN,2012.0,719.0,6.3,2.35,660;


Модой заполним признаки поскольку для них смое популярное значение не внесет большой разницы:
- aspect_ratio
- content_rating

In [1203]:
for col in ["aspect_ratio", "content_rating"]:
    data.fillna({col: data[col].mode()[0]}, inplace=True)

### Итак теперь остались только budget gross, в них больше всего пропусков и они самые важные 
- их не получится востановить по ксвенным признакам
- будем востанавлиать по медиане, но с группировкой по годам чтобы ценность валюты была одинаковоая

In [1204]:

data['gross'] = data.groupby('title_year')['gross'].transform(lambda x: x.fillna(x.median()))

data['budget'] = data.groupby('title_year')['budget'].transform(lambda x: x.fillna(x.median()))

In [1205]:
print(data[["gross", "budget"]].isna().sum())

gross     30
budget     0
dtype: int64


Как мы видим бюджет получилось заполнить, а вот валовый нет,
впринципе на 30 значений можно заполнить медианой

In [1206]:
data.fillna({'gross': data['gross'].median()}, inplace=True)

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000;
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0;
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000;
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000;
4,Color,Doug Walker,108.0,103.0,131.0,0.0,Rob Walker,131.0,20205006.5,Documentary,...,154.0,English,USA,R,25000000.0,2005.0,12.0,7.1,2.35,0;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4845,Color,Scott Smith,1.0,87.0,2.0,318.0,Daphne Zuniga,637.0,26903709.0,Comedy|Drama,...,6.0,English,Canada,R,20000000.0,2013.0,470.0,7.7,2.35,84;
4846,Color,Unknown,43.0,43.0,49.0,319.0,Valorie Curry,841.0,20205006.5,Crime|Drama|Mystery|Thriller,...,359.0,English,USA,TV-14,25000000.0,2005.0,593.0,7.5,16.00,32000;
4847,Color,Benjamin Roberds,13.0,76.0,0.0,0.0,Maxwell Moody,0.0,26903709.0,Drama|Horror|Thriller,...,3.0,English,USA,R,1400.0,2013.0,0.0,6.3,2.35,16;
4848,Color,Daniel Hsia,14.0,100.0,0.0,489.0,Daniel Henney,946.0,10443.0,Comedy|Drama|Romance,...,9.0,English,USA,PG-13,17000000.0,2012.0,719.0,6.3,2.35,660;


In [1207]:
(data.isna().sum()/data.shape[0]*100).sort_values(ascending=False)

color                        0.0
Director_Name                0.0
aspect_ratio                 0.0
imdb_score                   0.0
actor_2_facebook_likes       0.0
title_year                   0.0
budget                       0.0
content_rating               0.0
country                      0.0
language                     0.0
num_user_for_reviews         0.0
movie_imdb_link              0.0
plot_keywords                0.0
facenumber_in_poster         0.0
actor_3_name                 0.0
cast_total_facebook_likes    0.0
num_voted_users              0.0
movie_Title                  0.0
actor_1_name                 0.0
genres                       0.0
gross                        0.0
Actor_1_Facebook_likes       0.0
actor_2_name                 0.0
actor_3_Facebook_likes       0.0
director_Facebook_likes      0.0
duration                     0.0
num_Critic_for_reviews       0.0
movie_facebook_likes;        0.0
dtype: float64

Все теперь все пропуски заполненны

## План на преобразование данных

- преобразовать float стобцы в которых содержатся только целочисленные данные в int

### С помощью визуального EDA ответьте на следующие вопросы:
1. Как распределены рейтинги фильмов (IMDb score)?
2. Существует ли зависимость между бюджетом фильма и его кассовыми сборами?
3. Какие жанры встречаются чаще всего?
4. Влияет ли количество проголосовавших пользователей на итоговый рейтинг?
5. Есть ли связь между суммарными лайками актёров в Facebook и рейтингом
фильма?
6. Как изменялся средний бюджет фильмов по годам?
7. Какие страны лидируют по количеству произведённых фильмов?
8. Каково распределение длительности фильмов?
9. Различаются ли кассовые сборы в зависимости от возрастного рейтинга?
10. Кто из режиссёров снял больше всего фильмов в данной выборке?